# Masked Autoencoder — reconstruct what's hidden

**Foundation-model recipe** (the objective behind **MAE** and **VideoMAE**, lesson C3).

Hide half of an image's patches and train an encoder–decoder to reconstruct them. The model must learn structure to fill the gaps — a powerful self-supervised signal — and we report the reconstruction error on a **held-out test split**.

> Real data: scikit-learn `load_digits` (1,797 real 8×8 images). CPU is fine.

In [ ]:
import os, torch, torch.nn as nn, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 1200)); S, P = 8, 2; NP = (S // P) ** 2; MASK = 0.5; torch.manual_seed(0)

## 1 · Real digit images + patchify (16 patches of 2×2)

In [ ]:
d = load_digits()
X = torch.tensor(d.images / 16.0, dtype=torch.float32).unsqueeze(1)   # (1797,1,8,8)
Xtr, Xte = train_test_split(X, test_size=0.3, random_state=0)
Xtr, Xte = Xtr.to(device), Xte.to(device)
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape), "| patches/img", NP, "masked", int(MASK * NP))
def patchify(x): return x.unfold(2, P, P).unfold(3, P, P).reshape(x.shape[0], NP, P * P)
def unpatchify(p):
    g = S // P
    return p.reshape(p.shape[0], g, g, P, P).permute(0, 1, 3, 2, 4).reshape(p.shape[0], 1, S, S)

## 2 · Encoder–decoder over patch tokens

In [ ]:
class MAE(nn.Module):
    def __init__(self, dim=64):
        super().__init__(); self.emb = nn.Linear(P * P, dim); self.pos = nn.Parameter(torch.randn(1, NP, dim))
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(dim, 4, 128, batch_first=True), 2)
        self.dec = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, P * P))
    def forward(self, patches, mask):
        tok = self.emb(patches) + self.pos
        tok = tok * (~mask).unsqueeze(-1)                       # zero out masked tokens
        return self.dec(self.enc(tok))
model = MAE().to(device); opt = torch.optim.Adam(model.parameters(), 1e-3)

## 3 · Pretrain — reconstruct the masked patches

In [ ]:
hist = []
for step in range(STEPS + 1):
    idx = torch.randint(0, Xtr.shape[0], (128,)); p = patchify(Xtr[idx])
    mask = torch.rand(p.shape[0], NP, device=device) < MASK
    pred = model(p, mask); loss = (((pred - p) ** 2) * mask.unsqueeze(-1)).sum() / mask.sum() / (P * P)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0: hist.append((step, round(loss.item(), 4))); print(f"step {step:4d}  masked recon MSE {loss.item():.4f}")

## 4 · Held-out reconstruction error + a qualitative example

In [ ]:
with torch.no_grad():
    pte = patchify(Xte); m = torch.rand(Xte.shape[0], NP, device=device) < MASK
    test_mse = ((((model(pte, m) - pte) ** 2) * m.unsqueeze(-1)).sum() / m.sum() / (P * P)).item()
print(f"held-out masked-reconstruction MSE: {test_mse:.4f}")
x = Xte[:1]; p = patchify(x); mask = torch.rand(1, NP, device=device) < MASK
with torch.no_grad(): pred = model(p, mask)
masked_in = unpatchify(p * (~mask).unsqueeze(-1)); recon = unpatchify(torch.where(mask.unsqueeze(-1), pred, p))
fig, ax = plt.subplots(1, 4, figsize=(11, 3))
for a, im, ttl in zip(ax, [x, masked_in, unpatchify(pred), recon], ["input", "masked", "decoded", "filled-in"]):
    a.imshow(im[0, 0].cpu().clamp(0, 1), cmap="gray"); a.set_title(ttl); a.axis("off")
plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/B_mae_pretrain/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/B_mae_pretrain"; os.makedirs(run, exist_ok=True)
torch.save(model.state_dict(), f"{run}/mae.pt")
json.dump({"recon_mse": hist, "test_recon_mse": test_mse}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
The MAE objective underlies **C** VideoMAE and pretraining for **D** perception.

### Where to go next
- Extend masking across time → **VideoMAE** (tube masking, lesson C3).
- After pretraining, keep the encoder and fine-tune it for a downstream task.